# Baselines, comparison and extras

**Owner:** Teammate 4

Run this after the four model notebooks. It adds the two baselines and builds the comparison table and chart for the final slide.

Rules that apply to every notebook: use `prep` and `evaluate` for the split, the preparation and the scoring. Do not write your own. The test set is only touched inside `evaluate.run_feature_steps`.

In [ ]:
# Course      : CS 513 - Data Mining & Machine Learning
# First Name  : <fill in>
# Last Name   : <fill in>
# Id          : <fill in>
# Purpose     : Baselines, the comparison table and the beyond-class extras.
#
# AI NOTICE   : The starter code in this notebook was generated by AI (Claude).
#               Prompt and context: docs/AI_PROMPT_LOG.md, entry 1.
#               Mark any cell you add with AI help the same way.

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from src import evaluate, plots, prep

df = prep.load_table()
train, test = prep.split_by_building(df)
print(f"{len(df):,} filings, {df['bin'].nunique():,} buildings, "
      f"{df['is_unsafe'].mean():.1%} Unsafe")
print(f"Train: {len(train):,} rows   Test: {len(test):,} rows")

## Baselines every model has to beat

In [ ]:
evaluate.run_baselines(df).round(3)

## The comparison table

In [ ]:
table = pd.read_csv(config.COMPARISON_PATH)
view = table.pivot(index="model", columns="step", values="roc_auc").round(3)
print("Test ROC AUC by model and feature step")
display(view)
display(table[["model", "step", "roc_auc", "recall", "precision", "f1", "accuracy",
               "tp", "fp", "fn", "tn"]].round(3))

In [ ]:
fig, ax = plots.comparison_chart(table)
plt.show()

**Caveat for the slide.** The published 0.766 was measured on that project's own test split, not ours. Our fair comparison is our own step 1 (the same kind of features) against steps 2 and 3.

## Extra 1: SMOTE (beyond class)

SMOTE makes synthetic Unsafe examples. It runs inside the training folds only. Needs `pip install imbalanced-learn`.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

res_smote, _ = evaluate.run_feature_steps(
    "Random Forest + SMOTE",
    RandomForestClassifier(n_estimators=200, random_state=config.SEED),
    {"max_depth": [8, 12], "min_samples_leaf": [5, 20]},
    kind="plain", sampler=SMOTE(random_state=config.SEED), steps=["3_plus_violations"])
res_smote.round(3)

## To do (Teammate 4)

- Extra 2: threshold tuning. Pick the cut-off on the training folds that gives the recall the team wants, then report precision at that cut-off.
- Extra 3: SHAP on the Random Forest (`pip install shap`).
- Extra 4: a borough map from `map_lat` / `map_lon` in the table.
- Report results separately for buildings whose rating changed from the prior cycle. That is where the model has to earn its score.